In [1]:
import pandas as pd
import numpy as np
import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

from dotenv import load_dotenv

from sklearn.linear_model import LinearRegression
from scipy.stats import f_oneway

PATH = "./data/"
MODEL = "anthropic/claude-opus-4.6"
# MODEL = "gpt-3.5-turbo"

In [2]:
"""clients"""

load_dotenv()
llm_langfuse = get_client()
langfuse_handler = CallbackHandler()

LLM_KEY = os.getenv("LLM")  # Ensure this is set in .env
if not LLM_KEY:
    raise ValueError("LLM environment variable not set. Check your .env file.")

llm = ChatOpenAI(
    model=MODEL,
    api_key=LLM_KEY,
    base_url="https://openrouter.ai/api/v1",  # Remove if using OpenAI directly
    temperature=0.7,
)

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=LLM_KEY,
    base_url="https://openrouter.ai/api/v1",
)

In [3]:
from langchain.tools import tool
import pandas as pd
from scipy.stats import f_oneway
from sklearn.linear_model import LinearRegression

df = pd.read_csv("data/defects_data.csv")


@tool
def eda_tool():
    """
    Perform basic exploratory data analysis on the defects data.
    Returns summary statistics as a string.
    """
    return "Exploratory Data Analysis:\n" + df.describe(include="all").to_string()


@tool
def count_by_defect_type():
    """
    Count the number of defects for each defect type.
    """
    counts = df["defect_type"].value_counts().to_string()
    return f"Counts by defect type:\n{counts}"


@tool
def average_repair_cost_by_severity():
    """
    Calculate the average repair cost grouped by severity of defect.
    """
    df["repair_cost"] = pd.to_numeric(df["repair_cost"], errors="coerce")
    avg_cost = df.groupby("severity")["repair_cost"].mean()
    return f"Average repair cost by severity:\n{avg_cost.to_string()}"


@tool
def defects_over_time():
    """
    Get the count of defects over months.
    """
    df["defect_date"] = pd.to_datetime(df["defect_date"])
    monthly_counts = df.set_index("defect_date").resample("ME").size()
    return f"Defects over time (monthly):\n{monthly_counts.to_string()}"


@tool
def anova_tool():
    """
    Perform one-way ANOVA to test if repair_cost differs by defect_type.
    Returns: F-statistic and p-value as string.
    """
    df["repair_cost"] = pd.to_numeric(df["repair_cost"], errors="coerce")
    grouped = [group["repair_cost"].dropna() for _, group in df.groupby("defect_type")]
    if len(grouped) < 2:
        return "Not enough defect types for ANOVA."
    f_stat, p_val = f_oneway(*grouped)
    return f"ANOVA result for repair_cost by defect_type:\nF-statistic: {f_stat:.4f}, p-value: {p_val:.4e}"


@tool
def regression_tool():
    """
    Perform linear regression predicting repair_cost from defect_type and severity (one-hot encoded).
    Returns regression coefficients and model summary.
    """
    df["repair_cost"] = pd.to_numeric(df["repair_cost"], errors="coerce")
    X = pd.get_dummies(df[["defect_type", "severity"]], drop_first=True)
    y = df["repair_cost"]

    mask = y.notnull()
    X = X.loc[mask]
    y = y.loc[mask]

    model = LinearRegression()
    model.fit(X, y)
    coefs = dict(zip(X.columns, model.coef_))
    intercept = model.intercept_
    r2 = model.score(X, y)

    summary = (
        f"Linear Regression Results:\n"
        f"Intercept: {intercept:.2f}\n"
        f"R^2 Score: {r2:.4f}\n"
        f"Coefficients:\n" + "\n".join([f"  {k}: {v:.2f}" for k, v in coefs.items()])
    )
    return summary

In [4]:


agent = create_agent(
    llm,
    tools=[
        eda_tool,
        count_by_defect_type,
        average_repair_cost_by_severity,
        defects_over_time,
        anova_tool,
        regression_tool,
    ],
)

In [5]:
prompt = """
You are senior data scientist. 
You work through problems step-by-step using the tools available to you. 
Read the entire instructions carefully before you do anything else

## Workflow

### Step 1: Exploratory Data Analysis
Examine the dataset before doing anything else:
- Inspect shape, dtypes, missing values, and duplicates.
- Compute summary statistics for all numeric columns.
- Check cardinality and value distributions for categorical columns.
- Identify obvious data quality issues (outliers, implausible values, skew).

After completing EDA, state your key findings as a numbered list. For each finding, note whether it affects your downstream analysis choices (e.g., "Column X is heavily right-skewed → use a non-parametric test").

### Step 2: Analysis Plan
Based on your EDA findings and the user's question, propose a concrete analysis plan:
- List each analytical step you intend to take.
- Name the specific tool or statistical method for each step.
- State the assumption(s) each method requires and whether your EDA suggests they hold.

Do NOT execute the plan yet. Wait for the plan to be fully stated, then proceed.

### Step 3: Execution
Execute your plan one step at a time. After each tool call, interpret the result before moving on.

### Step 4: Report
Once all steps are complete, deliver a structured report using the exact format below.

## Output Format

Your final report MUST follow this structure exactly:
```
## Summary
[1–2 sentences: what you did and why.]

**Dataset context**: [Describe the dataset in plain language — what each row represents, the time range covered, total record count, and which columns were central to the analysis. Derive this from the data itself; do not parrot filenames or column headers without explanation.]

## EDA Findings
1. **[Finding title]**: [Result with key numbers]. [1–2 sentences of practical interpretation and whether it is statistically noteworthy.]
2. **[Finding title]**: ...

## Analysis Results
1. **[Analysis title]**
   - **Method**: [Name of test/model and why it was appropriate.]
   - **Key outputs**: [Test statistic, p-value, effect size, confidence interval — whichever apply.]
   - **Interpretation**: [1–2 sentences on practical meaning. Use language like "The difference of X units corresponds to a [small/medium/large] effect (Cohen's d = …)" rather than only "p < 0.05."]
2. **[Analysis title]**
   - **Method**: ...
   - **Key outputs**: ...
   - **Interpretation**: ...

## Limitations
- [Caveat 1: e.g., sample size concern, violated assumption, multiple comparisons, uncontrolled confounders.]
- [Caveat 2: ...]
```

## Output Example
```
## Summary
Compared conversion rates between the A and B variants of the checkout flow using 14 days of experiment data to determine whether variant B produces a meaningfully higher conversion rate.

**Dataset context**: Each row represents a single user session on the checkout page (n = 12,040). The data spans January 3–16, 2025 and includes the experiment variant assigned (`group`), whether the session resulted in a purchase (`converted`, binary), and the transaction amount (`revenue`, null for non-conversions). The analysis focused on the `group` and `converted` columns.

## EDA Findings
1. **Sample size per group**: Group A has 6,102 sessions; Group B has 5,938. The slight imbalance (50.7% / 49.3%) is negligible and does not require correction.
2. **Missing values**: The `revenue` column has 11,216 nulls (93.2%), corresponding to non-converting sessions. This is expected and not a data quality issue.
3. **Conversion rate distribution by day**: Daily conversion rates range from 2.8% to 4.1% with no visible time trend or day-of-week effect, supporting the assumption that observations are independent across days.

## Analysis Results
1. **Conversion rate comparison**
   - **Method**: Two-proportion z-test. Appropriate because the outcome is binary (converted / not converted) and both groups exceed n = 5,000, satisfying the normal approximation condition.
   - **Key outputs**: Group A conversion = 3.21%, Group B conversion = 3.74%, z = 1.82, p = 0.034 (one-tailed), absolute difference = +0.53 pp, 95% CI for difference = [+0.04 pp, +1.02 pp], relative lift = +16.5%.
   - **Interpretation**: Variant B's conversion rate is 0.53 percentage points higher, a relative lift of 16.5%. This is a small but operationally meaningful effect — at current traffic levels, it would translate to roughly 190 additional conversions per month. The confidence interval is wide, so the true lift could be as small as 0.04 pp.

## Limitations
- The experiment ran for only 14 days. Seasonal or promotional effects during this window may not generalize.
- Revenue per conversion was not analyzed; a higher conversion rate does not guarantee higher total revenue if average order value differs between groups.
- No correction for multiple comparisons was applied. If other metrics (e.g., bounce rate, time on page) are tested on the same data, the family-wise error rate increases.
```

## Rules
- Never invent data. If a tool call fails, report the failure and adjust your plan.
- Distinguish between statistical significance and practical significance in every interpretation.
- If the data does not support a requested analysis, explain why and suggest an alternative.

"""



In [6]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    config={"callbacks": [langfuse_handler]},
)

In [7]:
print(response["messages"][-1].content)

Both analyses are complete. Let me now interpret and compile the final report.

---

## Step 4: Report

---

## Summary
Analyzed a manufacturing defects dataset to determine whether repair costs differ meaningfully across defect types and severity levels, using one-way ANOVA and multiple linear regression.

**Dataset context**: Each row represents a single product defect record (n = 1,000) logged between January and June 2024 across approximately 100 distinct products. Key columns include `defect_type` (Structural, Functional, Cosmetic), `severity` (Critical, Minor, Moderate), `repair_cost` (continuous, in dollars), `defect_location` (Surface and two other categories), and `inspection_method` (Manual Testing and two others). The analysis focused on the relationship between `defect_type`, `severity`, and `repair_cost`.

## EDA Findings

1. **No missing data or duplicates**: All 1,000 records are complete across all 8 columns. No data cleaning was required, and the full dataset was used 